In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
T = 100
n = 8

# Initialize storage for results
P = np.zeros((T, n, n), dtype=np.float64)

# Define the transition matrix Pmat
Pmat = np.array([
    [90.81, 8.33, 0.68, 0.06, 0.08, 0.02, 0.01, 0.01],
    [0.70, 90.65, 7.79, 0.64, 0.06, 0.13, 0.02, 0.01],
    [0.09, 2.27, 91.05, 5.52, 0.74, 0.26, 0.01, 0.06],
    [0.02, 0.33, 5.95, 85.93, 5.30, 1.17, 1.12, 0.18],
    [0.03, 0.14, 0.67, 7.73, 80.53, 8.84, 1.00, 1.06],
    [0.01, 0.11, 0.24, 0.43, 6.48, 83.46, 4.07, 5.20],
    [0.21, 0.00, 0.22, 1.30, 2.38, 11.24, 64.86, 19.79],
    [0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 100.00]
]) / 100  # Convert percentages to probabilities

# Ratings labels
ratings = ['AAA', 'AA', 'A', 'BBB', 'BB', 'B', 'CCC', 'D']

# Set the initial matrix
P[0] = Pmat

# Compute P^n for n = 2 to 100
for t in range(1,T-1):
    P[t] = np.matmul(P[t-1],Pmat)

# Create 8 plots, one for each initial state
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for i in range(n):
    ax = axes[i]
    for j in range(n):
        ax.plot(range(T), P[:, i, j], label=f'{ratings[i]} to {ratings[j]}')
    ax.set_title(f'Transitions from {ratings[i]}')
    ax.set_xlabel('n')
    ax.set_ylabel('Probability')
    ax.grid(True)
    ax.legend(fontsize='small', loc='upper right')

plt.tight_layout()
plt.show()

In [ ]:
# Set seed for reproducibility
np.random.seed(13)

# Simulation of the Markov chain
K = 100  # Number of steps
Y = np.zeros(K, dtype=int)  # Array to hold state transitions

# Start with 'AAA' which corresponds to index 0
Y[0] = 0

# Simulation of transitions
for k in range(1, K):
    j = np.random.choice(range(8), p=Pmat[Y[k-1]])
    Y[k] = j

# Mapping indices back to ratings
Y_rated = [ratings[i] for i in Y]

# Plot the sequence of transitions
plt.step(range(K), Y_rated, where='post')
plt.xlabel('')
plt.ylabel('Bond Rating')
plt.title('100-Step Simulation of Bond Rating Transition')
plt.show()

In [ ]:
# Create a DataFrame for steps and transition probabilities
transition_data = []

for k in range(1, K):
    previous_state = ratings[Y[k-1]]
    next_state = ratings[Y[k]]
    transition_probability = Pmat[Y[k-1], Y[k]]
    transition_data.append({'Step': k, 'Previous State': previous_state, 'Next State': next_state, 
                            'Transition Probability': transition_probability})
    likelihood_sequence = np.prod(transition_probability)

pd.DataFrame(transition_data)

In [ ]:
n_transient = 7  # Number of transient states
H = Pmat[:n_transient, :n_transient]  # Top-left submatrix

Z = np.linalg.inv(np.identity(H.shape[0])-H)
# Compute the expected number of transitions
expected_transitions = Z.sum(axis=1)  # Sum of rows in Z

print("Expected Number of Transitions Before Absorption:")
for i, val in enumerate(expected_transitions):
    print(f"From State {i+1}: {val:.4f}")

In [ ]:
Q = np.zeros((n_transient, n_transient))  # Initialize Q matrix

# probabilities for transient states
for i in range(n_transient):
    for j in range(n_transient):
        if i == j:
            Q[i, j] = (Z[i, j] - 1) / Z[j, j]
        else:
            Q[i, j] = (Z[i, j]) / Z[j, j]

Q